In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/AB_US_2020.csv", low_memory=False)

# Work on a copy, keep df as your "checkpoint" if needed
df_clean = df.copy()

In [2]:
dup_id = df_clean[df_clean['id'].duplicated(keep=False)]
print(dup_id)

             id                                               name    host_id  \
15434  43806155  Beautiful 14th floor view mins from MGH - Evonify  212359760   
27734  43806155  Beautiful 14th floor view mins from MGH - Evonify  212359760   

      host_name neighbourhood_group   neighbourhood  latitude  longitude  \
15434   Evonify                 NaN        West End  42.36461  -71.06792   
27734   Evonify                 NaN  East Cambridge  42.36461  -71.06792   

             room_type  price  minimum_nights  number_of_reviews last_review  \
15434  Entire home/apt    222              13                  0         NaN   
27734  Entire home/apt    299              13                  0         NaN   

       reviews_per_month  calculated_host_listings_count  availability_365  \
15434                NaN                              45               150   
27734                NaN                               1                 0   

            city  
15434     Boston  
27734  Cambri

In [3]:
# If they're essentially identical, keep the first occurrence
df_clean = df_clean.drop_duplicates(subset='id', keep='first')

print(f"Rows before: {len(df)}, Rows after: {len(df_clean)}")

Rows before: 226030, Rows after: 226029


In [4]:
df_clean['minimum_nights'].sort_values(ascending=False).head(20)

196520    100000000
121846         1250
196514         1125
98220          1125
45259          1125
46094          1125
178099         1125
47771          1125
45561          1124
44590          1124
120716         1124
90377          1123
178711         1000
88138          1000
88146          1000
44445          1000
89345          1000
119475         1000
181110         1000
97515          1000
Name: minimum_nights, dtype: int64

In [5]:
df_clean = df_clean[df_clean['minimum_nights'] <= 365]
print(f"Rows after minimum_nights fix: {len(df_clean)}")

Rows after minimum_nights fix: 225964


In [6]:
# Remove $0 listings — not real bookable prices
df_clean = df_clean[df_clean['price'] > 0]

# Use percentile-based capping rather than an arbitrary number
upper_limit = df_clean['price'].quantile(0.99)
print(f"99th percentile price: {upper_limit}")

df_clean = df_clean[df_clean['price'] <= upper_limit]
print(f"Rows after price cleaning: {len(df_clean)}")

99th percentile price: 1700.0
Rows after price cleaning: 223667


In [7]:
# neighbourhood_group: structural, not random — don't fabricate it, just label it clearly
df_clean['neighbourhood_group'] = df_clean['neighbourhood_group'].fillna('Not Reported')

# last_review / reviews_per_month: missing because zero reviews exist — 0 is the correct fill, not "unknown"
df_clean['reviews_per_month'] = df_clean['reviews_per_month'].fillna(0)
# last_review stays missing — there's no fake date that would make sense here; we'll just remember why when we use it later

# name / host_name: negligible, safe to drop those few rows
df_clean = df_clean.dropna(subset=['name', 'host_name'])

In [8]:
df_clean['last_review'] = pd.to_datetime(df_clean['last_review'], errors='coerce')

/var/folders/xy/kfvm92f11z3481brx08h09mm0000gn/T/ipykernel_55157/1051192382.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean['last_review'] = pd.to_datetime(df_clean['last_review'], errors='coerce')


In [9]:
df_clean.to_csv("../data/processed/AB_US_2020_cleaned.csv", index=False)
print(f"Final cleaned shape: {df_clean.shape}")

Final cleaned shape: (223610, 17)
